In [ ]:
# ============================================================
# LST Threshold Lookup (STANDALONE)
# Loads directly from lst_zonal_stats.json + grid_reference.gpkg
# ============================================================
import os
import json
import operator
import numpy as np
import geopandas as gpd
from matplotlib.figure import Figure
from IPython.display import display

CONFIG_PATH = "config.json"

OPS = {
    ">=": operator.ge,
    "<=": operator.le,
    ">":  operator.gt,
    "<":  operator.lt,
    "==": operator.eq,
}


def load_zonal_stats_inputs(config_path=CONFIG_PATH, verbose=True):
    """Loads zonal stats JSON + grid reference geometry directly from disk."""
    with open(config_path) as f:
        config = json.load(f)

    out_cfg = config["output"]
    grid_cfg = config["grid_coverage"]
    GRID_ID_FIELD = grid_cfg["grid_id_field"]

    root_dir = out_cfg.get("root_dir") or os.path.dirname(os.path.normpath(config["input_folder"]))
    zonal_stats_dir = os.path.join(root_dir, out_cfg.get("zonal_stats_dirname", "lst_zonal_stats"))

    zonal_stats_json_path = os.path.join(zonal_stats_dir, "lst_zonal_stats.json")
    if verbose:
        print(f"Loading zonal stats JSON -> {zonal_stats_json_path}")
    with open(zonal_stats_json_path) as f:
        zonal_stats = json.load(f)

    grid_ids = np.array(zonal_stats["grid_ids"])
    scenes = zonal_stats["scenes"]
    if verbose:
        print(f"grid_ids={len(grid_ids)}  scenes={len(scenes)}")

    grid_reference_path = zonal_stats.get("grid_reference_path") or os.path.join(
        zonal_stats_dir, "grid_reference.gpkg"
    )
    if verbose:
        print(f"Loading grid geometry -> {grid_reference_path}")
    grid_geom = gpd.read_file(grid_reference_path)
    if verbose:
        print(f"grid_geom rows: {len(grid_geom)}")

    # Available seasons/day-night combos, for convenience
    seasons_available = sorted({s["season"] for s in scenes})
    if verbose:
        print(f"Seasons available: {seasons_available}")
        print(f"Day/night available: {sorted({s['day_night'] for s in scenes})}")

    return dict(
        config=config,
        GRID_ID_FIELD=GRID_ID_FIELD,
        root_dir=root_dir,
        zonal_stats_dir=zonal_stats_dir,
        zonal_stats=zonal_stats,
        grid_ids=grid_ids,
        scenes=scenes,
        grid_geom=grid_geom,
        seasons_available=seasons_available,
    )


print("=" * 60)
print("LOADING ZONAL STATS INPUTS")
print("=" * 60)
state = load_zonal_stats_inputs()
print("\nReady. `state` is loaded.")


# ============================================================
# Lookup function
# ============================================================

def lookup_lst_scenes(season, day_night, threshold, op, state=state,
                       stat_col="lst_mean", plot=True):
    """
    Find scenes/grids where LST meets a threshold.

    season     : e.g. "Summer" (see state['seasons_available'])
    day_night  : "day" or "night"
    threshold  : temperature in Kelvin
    op         : one of ">=", "<=", ">", "<", "=="
    stat_col   : "lst_mean" or "lst_median"
    """
    if op not in OPS:
        raise ValueError(f"op must be one of {list(OPS.keys())}")
    compare = OPS[op]

    scenes = state["scenes"]
    grid_ids = state["grid_ids"]
    grid_geom = state["grid_geom"]
    GRID_ID_FIELD = state["GRID_ID_FIELD"]

    results = []
    grid_hit_counts = {}  # grid_id -> # of matching scenes

    for scene in scenes:
        if scene["season"] != season or scene["day_night"] != day_night:
            continue

        vals = np.array(scene[stat_col], dtype=float)
        valid = ~np.isnan(vals)
        mask = valid & compare(vals, threshold)
        n_matching = int(mask.sum())

        if n_matching > 0:
            matching_ids = grid_ids[mask]
            results.append({
                "tiff_id": scene["tiff_id"],
                "datetime_local": scene["datetime_local"],
                "n_grids": n_matching,
                "grid_ids": matching_ids.tolist(),
                "max_val": float(np.nanmax(vals[mask])),
                "source_tiff_path": scene["source_tiff_path"],
            })
            for gid in matching_ids:
                grid_hit_counts[int(gid)] = grid_hit_counts.get(int(gid), 0) + 1

    results.sort(key=lambda r: r["datetime_local"])

    # ---- print list: datetime -> # grids fitting criteria ----
    print(f"Criteria: season={season}, day_night={day_night}, "
          f"{stat_col} {op} {threshold}K")
    print(f"Scenes matching: {len(results)}\n")
    print(f"{'datetime_local':<26} {'# grids':>8} {'max ('+stat_col+')':>18}   tiff_id")
    print("-" * 90)
    for r in results:
        print(f"{r['datetime_local']:<26} {r['n_grids']:>8} {r['max_val']:>18.2f}   {r['tiff_id']}")

    if not results:
        print("No matches found.")
        return results, grid_hit_counts

    # ---- plot: highlight matching grids, colored by hit count ----
    if plot:
        all_matching_ids = set(grid_hit_counts.keys())

        fig = Figure(figsize=(10, 10))
        ax = fig.subplots(1, 1)

        grid_geom.plot(ax=ax, color="whitesmoke", edgecolor="lightgrey", linewidth=0.3)

        highlight = grid_geom[grid_geom[GRID_ID_FIELD].isin(all_matching_ids)].copy()
        highlight["hit_count"] = highlight[GRID_ID_FIELD].map(grid_hit_counts)

        highlight.plot(
            ax=ax, column="hit_count", cmap="Reds", edgecolor="black", linewidth=0.4,
            legend=True, legend_kwds={"label": "# scenes matching", "shrink": 0.7},
        )

        ax.set_title(
            f"Grids where {season} {day_night} {stat_col} {op} {threshold}K\n"
            f"({len(all_matching_ids)} unique grid cell(s) across {len(results)} scene(s))",
            fontweight="bold"
        )
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect("equal")
        display(fig)

    return results, grid_hit_counts

LOADING ZONAL STATS INPUTS
Loading zonal stats JSON -> /Users/ks/Desktop/Wu/Summer26/02_SOLA_2018_2025/lst_zonal_stats/lst_zonal_stats.json


In [3]:
results, grid_hits = lookup_lst_scenes(
    season="Summer",
    day_night="night",
    threshold=330,      # tune down/up to zero in on the 344.88K spike
    op=">=",
    stat_col="lst_mean"
)

Criteria: season=Summer, day_night=night, lst_mean >= 330K
Scenes matching: 0

datetime_local              # grids     max (lst_mean)   tiff_id
------------------------------------------------------------------------------------------
No matches found.
